# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import requests

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
# For openAI
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far   ")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
openai = OpenAI()

# For ollama
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama-can pass anything here as it is local on your system')
requests.get("http://localhost:11434").content #print "Ollama is running" , otherwise open a terminal and run `ollama serve`

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
#prompt
q_system_prompt = """
You are a helpful assistant that can answer the question or list of questions provided by the user.
Make the answer concise and to the point.
Keep the answer strictly related to the topic.
"""

def q_user_prompt(question):
    user_prompt = f"""
Here is the question: {question}
"""
    return user_prompt


In [ ]:
# Get gpt-4o-mini to answer, with streaming
def get_gpt_answer(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": q_system_prompt},
            {"role": "user", "content": q_user_prompt(question)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''    
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# Get Llama 3.2 to answer

def get_llama_answer(question):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": q_system_prompt},
            {"role": "user", "content": q_user_prompt(question)}
        ],
        stream=True)
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# Get user input for question
def get_user_question():
    my_question = input("Please enter your question:")
    print(f"User's question is : {my_question}")
    return my_question

In [ ]:
# Call the function
get_gpt_answer("What is the capital of France , population with density, largest provinve name, currency and language?") # test with General Knowledge
get_gpt_answer("How do you define : LangChain vs LiteLLM: How to Choose the Right LLM Framework") # test with static technical Query

get_gpt_answer(get_user_question()) # get user input of question


In [ ]:
# Call the function
get_llama_answer("What is the capital of France , population with density, largest provinve name, currency and language?") # test with General Knowledge
get_llama_answer("How do you define : LangChain vs LiteLLM: How to Choose the Right LLM Framework") # test with static technical Query
get_llama_answer(get_user_question()) # get user input of question